# LR

In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [2]:
DATA_PATH = r"D:\墨大sml作业\FeatureA_Repeated"
OUTPUT_PATH = r"D:\墨大sml作业\LR_FeatureA_Results"

os.makedirs(OUTPUT_PATH, exist_ok=True)

N_REPEATS = 10

C_VALUES = [0.1, 1, 10]

N_INNER_REPEATS = 3
VALID_RATIO = 0.2
BASE_SEED = 42

In [3]:
feature_cols = [
    "user_avg_rating",
    "user_rating_count",
    "user_rating_std",
    "user_like_count",
    "user_like_ratio",
    "user_rating_timespan",
    "user_avg_gap_days",

    "item_avg_rating",
    "item_rating_count",
    "item_rating_std",
    "item_like_count",
    "item_like_ratio",

    "global_mean",
    "movie_age_at_rating"
]

In [4]:
def stratified_split_from_scratch(df, label_col, test_ratio=0.2, random_seed=42):
    rng = np.random.default_rng(random_seed)

    train_indices = []
    test_indices = []

    for label_value in df[label_col].unique():
        label_indices = df[df[label_col] == label_value].index.to_numpy()
        rng.shuffle(label_indices)

        test_size = int(len(label_indices) * test_ratio)

        test_indices.extend(label_indices[:test_size])
        train_indices.extend(label_indices[test_size:])

    train_df = df.loc[train_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    test_df = df.loc[test_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    return train_df, test_df

In [5]:
def compute_basic_metrics_from_scratch(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

In [6]:
def compute_auc_from_scratch(y_true, y_prob):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    sorted_indices = np.argsort(-y_prob)
    y_true_sorted = y_true[sorted_indices]

    pos_count = np.sum(y_true == 1)
    neg_count = np.sum(y_true == 0)

    if pos_count == 0 or neg_count == 0:
        return 0

    tp = 0
    fp = 0

    tpr_list = [0]
    fpr_list = [0]

    for label in y_true_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1

        tpr_list.append(tp / pos_count)
        fpr_list.append(fp / neg_count)

    auc = 0

    for i in range(1, len(tpr_list)):
        auc += (
            (fpr_list[i] - fpr_list[i - 1])
            *
            (tpr_list[i] + tpr_list[i - 1])
            / 2
        )

    return auc

In [7]:
def train_and_evaluate_lr(train_df, test_df, C_value):
    X_train = train_df[feature_cols]
    y_train = train_df["label"]

    X_test = test_df[feature_cols]
    y_test = test_df["label"]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(
        C=C_value,
        penalty="l2",
        solver="liblinear",
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    metrics = compute_basic_metrics_from_scratch(y_test, y_pred)
    metrics["auc"] = compute_auc_from_scratch(y_test, y_prob)

    return metrics

In [8]:
def tune_lr_C_from_scratch(train_df, C_values, n_inner_repeats=3, valid_ratio=0.2, base_seed=100):
    tuning_records = []

    for C_value in C_values:
        inner_f1_scores = []

        for inner_id in range(n_inner_repeats):
            inner_train_df, valid_df = stratified_split_from_scratch(
                train_df,
                label_col="label",
                test_ratio=valid_ratio,
                random_seed=base_seed + inner_id
            )

            metrics = train_and_evaluate_lr(
                train_df=inner_train_df,
                test_df=valid_df,
                C_value=C_value
            )

            inner_f1_scores.append(metrics["f1"])

        tuning_records.append({
            "C": C_value,
            "mean_validation_f1": np.mean(inner_f1_scores),
            "std_validation_f1": np.std(inner_f1_scores, ddof=1)
        })

    tuning_df = pd.DataFrame(tuning_records)

    best_C = tuning_df.sort_values(
        by="mean_validation_f1",
        ascending=False
    ).iloc[0]["C"]

    return best_C, tuning_df

In [9]:
all_results = []
all_tuning_results = []

for repeat_id in range(1, N_REPEATS + 1):

    print("=" * 60)
    print(f"Outer Repeat {repeat_id:02d}")
    print("=" * 60)

    repeat_folder = os.path.join(
        DATA_PATH,
        f"repeat_{repeat_id:02d}"
    )

    train_path = os.path.join(repeat_folder, "feature_A_train.csv")
    test_path = os.path.join(repeat_folder, "feature_A_test.csv")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    print("Train shape:", train_df.shape)
    print("Test shape:", test_df.shape)

    best_C, tuning_df = tune_lr_C_from_scratch(
        train_df=train_df,
        C_values=C_VALUES,
        n_inner_repeats=N_INNER_REPEATS,
        valid_ratio=VALID_RATIO,
        base_seed=1000 + repeat_id * 10
    )

    print("Best C:", best_C)

    tuning_df["outer_repeat"] = repeat_id
    all_tuning_results.append(tuning_df)

    final_metrics = train_and_evaluate_lr(
        train_df=train_df,
        test_df=test_df,
        C_value=best_C
    )

    result_row = {
        "outer_repeat": repeat_id,
        "best_C": best_C,
        **final_metrics
    }

    all_results.append(result_row)

    print("Accuracy :", round(final_metrics["accuracy"], 4))
    print("Precision:", round(final_metrics["precision"], 4))
    print("Recall   :", round(final_metrics["recall"], 4))
    print("F1       :", round(final_metrics["f1"], 4))
    print("AUC      :", round(final_metrics["auc"], 4))

Outer Repeat 01
Train shape: (16000211, 17)
Test shape: (4000052, 17)
Best C: 1.0
Accuracy : 0.7138
Precision: 0.7098
Recall   : 0.7231
F1       : 0.7163
AUC      : 0.7876
Outer Repeat 02
Train shape: (16000211, 17)
Test shape: (4000052, 17)
Best C: 1.0
Accuracy : 0.7137
Precision: 0.7096
Recall   : 0.723
F1       : 0.7163
AUC      : 0.7877
Outer Repeat 03
Train shape: (16000211, 17)
Test shape: (4000052, 17)
Best C: 1.0
Accuracy : 0.7141
Precision: 0.7102
Recall   : 0.723
F1       : 0.7166
AUC      : 0.788
Outer Repeat 04
Train shape: (16000211, 17)
Test shape: (4000052, 17)
Best C: 0.1
Accuracy : 0.7136
Precision: 0.7096
Recall   : 0.7228
F1       : 0.7161
AUC      : 0.7875
Outer Repeat 05
Train shape: (16000211, 17)
Test shape: (4000052, 17)
Best C: 0.1
Accuracy : 0.7135
Precision: 0.7095
Recall   : 0.7224
F1       : 0.7159
AUC      : 0.7874
Outer Repeat 06
Train shape: (16000211, 17)
Test shape: (4000052, 17)
Best C: 0.1
Accuracy : 0.7139
Precision: 0.7101
Recall   : 0.7227
F1     

C:\Users\MJ\AppData\Roaming\Python\Python39\site-packages\sklearn\preprocessing\_data.py:1037: RuntimeWarning: invalid value encountered in sqrt
  np.sqrt(self.var_), copy=False, constant_mask=constant_mask
C:\Users\MJ\AppData\Roaming\Python\Python39\site-packages\sklearn\preprocessing\_data.py:1037: RuntimeWarning: invalid value encountered in sqrt
  np.sqrt(self.var_), copy=False, constant_mask=constant_mask
C:\Users\MJ\AppData\Roaming\Python\Python39\site-packages\sklearn\preprocessing\_data.py:1037: RuntimeWarning: invalid value encountered in sqrt
  np.sqrt(self.var_), copy=False, constant_mask=constant_mask
C:\Users\MJ\AppData\Roaming\Python\Python39\site-packages\sklearn\preprocessing\_data.py:1037: RuntimeWarning: invalid value encountered in sqrt
  np.sqrt(self.var_), copy=False, constant_mask=constant_mask
C:\Users\MJ\AppData\Roaming\Python\Python39\site-packages\sklearn\preprocessing\_data.py:1037: RuntimeWarning: invalid value encountered in sqrt
  np.sqrt(self.var_), copy=

Best C: 0.1
Accuracy : 0.7136
Precision: 0.7097
Recall   : 0.7225
F1       : 0.716
AUC      : 0.7876
Outer Repeat 10
Train shape: (16000211, 17)
Test shape: (4000052, 17)
Best C: 10.0
Accuracy : 0.7139
Precision: 0.71
Recall   : 0.7228
F1       : 0.7163
AUC      : 0.7876


In [10]:
results_df = pd.DataFrame(all_results)
tuning_results_df = pd.concat(all_tuning_results, ignore_index=True)

results_path = os.path.join(OUTPUT_PATH, "LR_FeatureA_repeated_results.csv")
tuning_path = os.path.join(OUTPUT_PATH, "LR_FeatureA_tuning_results.csv")

results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
tuning_results_df.to_csv(tuning_path, index=False, encoding="utf-8-sig")

print("Saved repeated test results to:")
print(results_path)

print("Saved tuning results to:")
print(tuning_path)

results_df

Saved repeated test results to:
D:\墨大sml作业\LR_FeatureA_Results\LR_FeatureA_repeated_results.csv
Saved tuning results to:
D:\墨大sml作业\LR_FeatureA_Results\LR_FeatureA_tuning_results.csv


,outer_repeat,best_C,accuracy,precision,recall,f1,tp,tn,fp,fn,auc
0,1,1.0,0.713818,0.709753,0.723051,0.716340,1445438,1409871,591099,553644,0.787602
1,2,1.0,0.713715,0.709615,0.723037,0.716263,1445410,1409487,591483,553672,0.787670
2,3,1.0,0.714150,0.710247,0.722976,0.716555,1445288,1411348,589622,553794,0.787958
3,4,0.1,0.713615,0.709586,0.722769,0.716117,1444874,1409624,591346,554208,0.787549
4,5,0.1,0.713470,0.709545,0.722379,0.715905,1444095,1409824,591146,554987,0.787399
5,6,0.1,0.713921,0.710060,0.722656,0.716303,1444649,1411074,589896,554433,0.787713
6,7,0.1,0.713513,0.709649,0.722270,0.715904,1443877,1410213,590757,555205,0.787478
7,8,1.0,0.713911,0.709820,0.723203,0.716449,1445743,1409938,591032,553339,0.787826
8,9,0.1,0.713636,0.709742,0.722462,0.716045,1444260,1410321,590649,554822,0.787576
9,10,10.0,0.713916,0.709988,0.722812,0.716343,1444960,1410742,590228,554122,0.787574


In [11]:
summary_records = []

for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
    values = results_df[metric].values

    summary_records.append({
        "metric": metric,
        "mean": np.mean(values),
        "std": np.std(values, ddof=1),
        "standard_error": np.std(values, ddof=1) / np.sqrt(len(values))
    })

summary_df = pd.DataFrame(summary_records)

summary_path = os.path.join(OUTPUT_PATH, "LR_FeatureA_summary.csv")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

summary_df

,metric,mean,std,standard_error
0,accuracy,0.713767,0.000214,0.000068
1,precision,0.709801,0.000230,0.000073
2,recall,0.722761,0.000315,0.000100
3,f1,0.716222,0.000222,0.000070
4,auc,0.787635,0.000165,0.000052


In [12]:
best_C_frequency = results_df["best_C"].value_counts().reset_index()
best_C_frequency.columns = ["C", "frequency"]

best_C_frequency_path = os.path.join(OUTPUT_PATH, "LR_best_C_frequency.csv")
best_C_frequency.to_csv(best_C_frequency_path, index=False, encoding="utf-8-sig")

best_C_frequency

,C,frequency
0,0.1,5
1,1.0,4
2,10.0,1
